# HyperQwen End-to-End: LIMO LoRA + PWMV on Qwen3.6-35B-A3B

Self-contained notebook — trains LoRA, uploads adapter IMMEDIATELY after training (crash-safe), then evaluates 4 conditions and uploads results.

## Pipeline

1. Setup + auth
2. Load Qwen3.6-35B-A3B base (~4 min)
3. Load + filter s1K-1.1 (max_seq=6144, ~200-500 samples)
4. PEFT LoRA wrap (r=16, targets: GA + GDN + shared_expert)
5. Train 3 epochs (~16 min, 80+ steps)
6. **Save + upload adapter to HF IMMEDIATELY** (crash safety)
7. Load qwen-honest probe
8. Run 4-way eval: base-greedy / adapter-greedy / base-PWMV / adapter-PWMV (N=15, ~2h)
9. Upload full results + charts

Target: base-greedy 0.70 → adapter-PWMV ≥ 0.76 (stacking gain).

Total budget: ~3h on RTX 6000 Blackwell.

## Cell 1 — Setup

In [ ]:
import sys, subprocess, os, shutil
from pathlib import Path
def pip(*a): return subprocess.run([sys.executable, '-m', 'pip', *a], check=False)

try:
    import transformers
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    ok = 'qwen3_5_moe' in CONFIG_MAPPING_NAMES
except Exception: ok = False

if not ok:
    pip('install', '-q', 'accelerate', 'datasets', 'huggingface_hub==1.5.0',
        'safetensors', 'einops', 'scikit-learn', 'sentencepiece', 'tokenizers',
        'protobuf', 'scipy', 'hf_transfer', 'peft', 'trl')
    pip('uninstall', '-y', 'transformers', 'causal-conv1d')
    SRC = '/content/transformers_src'
    if Path(SRC).exists(): shutil.rmtree(SRC)
    subprocess.run(['git','clone','--quiet','--depth=1',
                    'https://github.com/huggingface/transformers.git', SRC], check=True)
    pip('install','--force-reinstall','--no-deps','--no-cache-dir', SRC)
    pip('install','--no-cache-dir','flash-linear-attention')
    for m in list(sys.modules):
        if m.startswith('transformers') or m.startswith('huggingface_hub'): del sys.modules[m]

pip('install', '-q', '--upgrade', 'torchao>=0.16.0', 'peft', 'trl', 'datasets')

try:
    from google.colab import userdata
    t = userdata.get('HF_TOKEN')
    if t:
        from huggingface_hub import login
        login(token=t); print('HF auth OK')
except: pass

import json, re, time, random, pickle
import numpy as np
import torch
import torch.nn as nn
from tqdm.auto import tqdm
from collections import defaultdict, Counter
from datasets import load_dataset, Dataset
from safetensors import safe_open
from huggingface_hub import snapshot_download, HfApi, create_repo
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

OUT = Path('/content/hyperqwen'); OUT.mkdir(exist_ok=True)
HF_ADAPTER_REPO = 'caiovicentino1/Qwen3.6-35B-A3B-LIMO-Probe'
HF_STAGE_B = 'caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b'
HF_DEEPCONF = 'caiovicentino1/qwen36-deepconf-probe'
MODEL_ID = 'Qwen/Qwen3.6-35B-A3B'

# Training hyperparams
N_SAMPLES = 500
MAX_SEQ_LEN = 6144
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LR = 1e-4
EPOCHS = 3
BATCH_SIZE = 1
GRAD_ACCUM = 8

# Eval hyperparams
N_EVAL = 15
EVAL_SEED = 123
N_TRACES = 4
TEMP = 0.7
MAX_NEW = 3500
FORCE_SUFFIX = '\n</think>\n\nFinal answer: \\boxed{'

print('setup done')

## Cell 2 — Load Qwen3.6 base

In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map='cuda',
    attn_implementation='sdpa', trust_remote_code=True)
model.config.use_cache = False
model.gradient_checkpointing_enable()
print(f'Model loaded: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# Find LoRA target modules
target_set = set()
skip_patterns = ['in_proj_a', 'in_proj_b', 'routed_experts', 'norm', 'embed']
for n, m in model.named_modules():
    if not isinstance(m, torch.nn.Linear): continue
    if any(sk in n for sk in skip_patterns): continue
    lname = n.split('.')[-1]
    if lname in ['q_proj','k_proj','v_proj','o_proj','in_proj_qkv','out_proj',
                 'gate_proj','up_proj','down_proj']:
        if lname in ['gate_proj','up_proj','down_proj'] and 'shared_expert' not in n:
            continue
        target_set.add(lname)

TARGET_MODULES = sorted(target_set)
print(f'LoRA targets ({len(TARGET_MODULES)}): {TARGET_MODULES}')

## Cell 3 — Load + format s1K-1.1 dataset

In [ ]:
ds = load_dataset('simplescaling/s1K-1.1', split='train')
print(f's1K-1.1: {len(ds)} samples')

def format_s1k(ex):
    q = ex.get('question', '')
    think = ''
    for k in ['deepseek_thinking_trajectory', 'gemini_thinking_trajectory']:
        v = ex.get(k)
        if v: think = v[0] if isinstance(v, list) else v; break
    ans = ex.get('deepseek_attempt') or ex.get('gemini_attempt') or ex.get('solution') or ''
    if not q or not think or not ans: return None
    assistant = f'<think>\n{think.strip()}\n</think>\n\n{ans.strip()}'
    msgs = [{'role':'user','content':q.strip()}, {'role':'assistant','content':assistant}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False, enable_thinking=False)

formatted = []
for ex in ds:
    text = format_s1k(ex)
    if not text: continue
    n = len(tok.encode(text, add_special_tokens=False))
    if n < 200 or n > MAX_SEQ_LEN: continue
    formatted.append({'text': text, 'n_tokens': n})
    if len(formatted) >= N_SAMPLES: break

print(f'{len(formatted)} samples pass filter (avg {np.mean([s["n_tokens"] for s in formatted]):.0f} tokens)')
train_ds = Dataset.from_list([{'text': s['text']} for s in formatted])

## Cell 4 — PEFT LoRA wrap

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES, lora_dropout=LORA_DROPOUT,
    bias='none', task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Cell 5 — Train LoRA (~16-25 min depending on dataset size)

In [ ]:
from trl import SFTConfig, SFTTrainer

args = SFTConfig(
    output_dir=str(OUT / 'ckpt'),
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LR, lr_scheduler_type='cosine', warmup_ratio=0.05,
    bf16=True, logging_steps=5, save_steps=200, save_total_limit=1,
    report_to='none', optim='adamw_torch',
    max_length=MAX_SEQ_LEN, packing=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    dataset_text_field='text', dataloader_num_workers=0,
    weight_decay=0.01,
)
trainer = SFTTrainer(model=model, args=args, train_dataset=train_ds, processing_class=tok)

total_steps = len(train_ds) * EPOCHS // (BATCH_SIZE * GRAD_ACCUM)
print(f'Training {total_steps} steps starting at {time.ctime()}')
t0 = time.time()
result = trainer.train()
print(f'Training done in {(time.time()-t0)/60:.1f} min, final loss: {result.training_loss:.4f}')

# SAVE ADAPTER NOW
adapter_dir = OUT / 'lora_adapter'
trainer.save_model(str(adapter_dir))
tok.save_pretrained(str(adapter_dir))
print(f'Adapter saved locally: {adapter_dir}')

## Cell 6 — 🚨 CRASH-SAFE UPLOAD: push adapter to HF immediately

In [ ]:
# UPLOAD IMMEDIATELY so crash doesn't lose work
api = HfApi()
create_repo(HF_ADAPTER_REPO, repo_type='model', private=False, exist_ok=True)
api.upload_folder(
    folder_path=str(OUT / 'lora_adapter'),
    path_in_repo='lora_adapter',
    repo_id=HF_ADAPTER_REPO, repo_type='model',
    commit_message=f'LIMO LoRA r={LORA_R}, {len(train_ds)} samples, {EPOCHS} epochs'
)
print(f'✅ Adapter uploaded: https://huggingface.co/{HF_ADAPTER_REPO}/tree/main/lora_adapter')
print(f'   Training loss: {result.training_loss:.4f}')

## Cell 7 — Load qwen-honest probe + install L11 hook

In [ ]:
probe_dir = snapshot_download(HF_DEEPCONF, repo_type='model', cache_dir='/content/cache',
                               allow_patterns=['probe_l11.pkl'])
with open(Path(probe_dir)/'probe_l11.pkl', 'rb') as f:
    probe = pickle.load(f)
shutil.copy(Path(probe_dir)/'probe_l11.pkl', OUT / 'probe_l11.pkl')
print(f'probe loaded (AUROC ~0.78)')

captured_l11 = {'chunks': []}
def l11_hook(m, i, o):
    h = o[0] if isinstance(o, tuple) else o
    captured_l11['chunks'].append(h[:, -1, :].detach().float().cpu())

base_ref = model.base_model.model if hasattr(model, 'base_model') else model
layer_L = None
for path in [('language_model','layers'), ('model','language_model','layers'), ('model','layers')]:
    obj = base_ref; ok = True
    try:
        for p in path: obj = getattr(obj, p)
        layer_L = obj[11]; break
    except: continue
assert layer_L is not None, 'L11 not found'
h_handle = layer_L.register_forward_hook(l11_hook)
print(f'OK L11 hook installed on {type(layer_L).__name__}')

## Cell 8 — Load eval set (seed 123, same as original PWMV eval)

In [ ]:
corpus = snapshot_download(HF_STAGE_B, repo_type='dataset', cache_dir='/content/cache')
shards = sorted((Path(corpus)/'shards').glob('q*.safetensors'))

questions = []
seen = set()
for shard in shards:
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        q = meta['question']
        if q in seen: continue
        seen.add(q)
        opts = json.loads(meta['options'])
        if len(opts) != 10: continue
        questions.append(dict(question=q, options=opts, gold_letter=meta['gold_letter']))

random.seed(EVAL_SEED); random.shuffle(questions)
eval_set = questions[:N_EVAL]
print(f'{len(eval_set)} eval prompts (seed={EVAL_SEED})')

## Cell 9 — Eval helpers

In [ ]:
def extract_answer(text):
    post = text.split('</think>')[-1] if '</think>' in text else text
    for pattern in [
        r'\\boxed\{([A-J])\}',
        r'[Aa]nswer\s*(?:is\s*)?[:=]?\s*\*?\*?\(?([A-J])\)?\*?\*?',
        r'^\s*\(?([A-J])\)?\s*\.?\s*$',
    ]:
        m = re.search(pattern, post, re.MULTILINE)
        if m: return m.group(1).upper()
    m = re.findall(r'\\boxed\{([A-J])\}', text)
    if m: return m[-1]
    tail = post[-300:] if post else text[-300:]
    letters = re.findall(r'\b([A-J])\b', tail)
    return letters[-1] if letters else None

def fmt(q, opts):
    choices = '\n'.join(f'{chr(65+i)}. {o}' for i, o in enumerate(opts))
    content = (f'Answer the following multiple-choice question. '
               f'Reason step by step, then put the letter in \\boxed{{}}.\n\n'
               f'Question: {q}\n\nOptions:\n{choices}')
    return tok.apply_chat_template([{'role':'user','content':content}],
                                    tokenize=False, add_generation_prompt=True, enable_thinking=True)

def force_close(full_ids, prompt_len):
    gen = tok.decode(full_ids[prompt_len:].tolist(), skip_special_tokens=False)
    if '</think>' in gen:
        return tok.decode(full_ids[prompt_len:].tolist(), skip_special_tokens=True)
    full = tok.decode(full_ids.tolist(), skip_special_tokens=False) + FORCE_SUFFIX
    ctx = tok(full, return_tensors='pt', add_special_tokens=False).input_ids.cuda()
    with torch.no_grad():
        out = model.generate(ctx, max_new_tokens=15, do_sample=False,
                             pad_token_id=tok.pad_token_id, use_cache=True)
    suf = tok.decode(out[0, ctx.shape[1]:].tolist(), skip_special_tokens=True)
    return tok.decode(full_ids[prompt_len:].tolist(), skip_special_tokens=True) + FORCE_SUFFIX + suf

def gen_greedy(ids):
    captured_l11['chunks'] = []
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=MAX_NEW, do_sample=False,
                             pad_token_id=tok.pad_token_id, use_cache=True)
    return extract_answer(force_close(out[0], ids.shape[1]))

def gen_pwmv(ids):
    captured_l11['chunks'] = []
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=MAX_NEW, do_sample=True, temperature=TEMP,
                             num_return_sequences=N_TRACES, top_p=0.95,
                             pad_token_id=tok.pad_token_id, use_cache=True)
    chunks = captured_l11['chunks']
    if len(chunks) >= 2:
        embs = torch.stack(chunks[1:], dim=0).mean(dim=0).numpy()
        scores = probe.predict_proba(embs)[:, 1].tolist()
    else:
        scores = [1.0] * N_TRACES
    answers = [extract_answer(force_close(out[i], ids.shape[1])) for i in range(N_TRACES)]
    weighted = defaultdict(float)
    for a, s in zip(answers, scores):
        if a: weighted[a] += s
    return max(weighted, key=weighted.get) if weighted else None

print('helpers ready')

## Cell 10 — Run 4-way eval with checkpoint saving (crash-safe)

In [ ]:
results = {'base_greedy': [], 'adapter_greedy': [], 'base_pwmv': [], 'adapter_pwmv': []}
model.eval()
t0 = time.time()

for qi, q in enumerate(tqdm(eval_set, desc='4-way eval')):
    p = fmt(q['question'], q['options'])
    ids = tok(p, return_tensors='pt').input_ids.cuda()
    if ids.shape[1] > 1536: continue
    gold = q['gold_letter']

    # ADAPTER enabled (default state)
    try:
        ag = gen_greedy(ids); results['adapter_greedy'].append((ag, ag == gold))
    except Exception as e:
        results['adapter_greedy'].append((None, False))
    try:
        ap = gen_pwmv(ids); results['adapter_pwmv'].append((ap, ap == gold))
    except Exception as e:
        results['adapter_pwmv'].append((None, False))

    # BASE (disable adapter)
    with model.disable_adapter():
        try:
            bg = gen_greedy(ids); results['base_greedy'].append((bg, bg == gold))
        except Exception as e:
            results['base_greedy'].append((None, False))
        try:
            bp = gen_pwmv(ids); results['base_pwmv'].append((bp, bp == gold))
        except Exception as e:
            results['base_pwmv'].append((None, False))

    # Checkpoint every 3 prompts — crash safe
    if (qi+1) % 3 == 0:
        print(f'  [{qi+1}/{len(eval_set)}]', end=' ')
        for k in ['base_greedy','adapter_greedy','base_pwmv','adapter_pwmv']:
            print(f'{k}={np.mean([r[1] for r in results[k]]):.3f}', end=' ')
        print()
        # Save partial results to disk
        partial = {k: [(r[0], bool(r[1])) for r in v] for k, v in results.items()}
        with open(OUT / 'eval_partial.json', 'w') as f:
            json.dump(partial, f, indent=2)

h_handle.remove()

print(f'\n=== FINAL (n={len(eval_set)}, {(time.time()-t0)/60:.1f} min) ===')
for k in ['base_greedy','base_pwmv','adapter_greedy','adapter_pwmv']:
    acc = np.mean([r[1] for r in results[k]])
    nones = sum(1 for r in results[k] if r[0] is None)
    print(f'{k:20s}: acc={acc:.3f}  (None: {nones})')

b_g = np.mean([r[1] for r in results['base_greedy']])
b_p = np.mean([r[1] for r in results['base_pwmv']])
a_g = np.mean([r[1] for r in results['adapter_greedy']])
a_p = np.mean([r[1] for r in results['adapter_pwmv']])
print(f'\n=== DELTAS ===')
print(f'PWMV on base:     {(b_p-b_g)*100:+.1f}pp')
print(f'PWMV on adapter:  {(a_p-a_g)*100:+.1f}pp')
print(f'LoRA on greedy:   {(a_g-b_g)*100:+.1f}pp')
print(f'LoRA on PWMV:     {(a_p-b_p)*100:+.1f}pp')
print(f'FULL STACK:       {(a_p-b_g)*100:+.1f}pp (adapter+PWMV vs base greedy)')

## Cell 11 — Upload all artifacts + wrapper code + README

In [ ]:
# Write wrapper code
WRAPPER_CODE = '''"""HyperQwen-A3B wrapper: LIMO LoRA + probe-weighted self-consistency."""
import torch, re, pickle
from collections import defaultdict
from transformers import AutoTokenizer, AutoModelForImageTextToText
from peft import PeftModel

class HyperQwenWrapper:
    def __init__(self, base_id="Qwen/Qwen3.6-35B-A3B", adapter_path=None, probe_path=None):
        self.tok = AutoTokenizer.from_pretrained(base_id, trust_remote_code=True)
        if self.tok.pad_token_id is None: self.tok.pad_token = self.tok.eos_token
        base = AutoModelForImageTextToText.from_pretrained(
            base_id, dtype=torch.bfloat16, device_map="cuda",
            attn_implementation="sdpa", trust_remote_code=True)
        self.model = PeftModel.from_pretrained(base, adapter_path) if adapter_path else base
        self.model.eval()
        self.probe = pickle.load(open(probe_path, "rb")) if probe_path else None
        self._chunks = []
        self._install_hook()

    def _install_hook(self):
        def hook(m, i, o):
            h = o[0] if isinstance(o, tuple) else o
            self._chunks.append(h[:, -1, :].detach().float().cpu())
        base = self.model.base_model.model if hasattr(self.model, "base_model") else self.model
        for path in [("language_model","layers"), ("model","language_model","layers"), ("model","layers")]:
            obj = base
            try:
                for p in path: obj = getattr(obj, p)
                layer = obj[11]; break
            except: continue
        self.handle = layer.register_forward_hook(hook)

    @staticmethod
    def extract_answer(text):
        post = text.split("</think>")[-1] if "</think>" in text else text
        m = re.search(r"\\boxed\{([A-J])\}", post)
        return m.group(1) if m else None

    def predict(self, question, options, n_traces=4, temperature=0.7, max_new=3500):
        choices = "\n".join(f"{chr(65+i)}. {o}" for i, o in enumerate(options))
        content = ("Answer the following multiple-choice question. "
            "Reason step by step, then put the letter in \\boxed{}.\n\n"
            f"Question: {question}\n\nOptions:\n{choices}")
        prompt = self.tok.apply_chat_template(
            [{"role":"user","content":content}],
            tokenize=False, add_generation_prompt=True, enable_thinking=True)
        ids = self.tok(prompt, return_tensors="pt").input_ids.cuda()
        self._chunks = []
        with torch.no_grad():
            out = self.model.generate(
                ids, max_new_tokens=max_new, do_sample=True, temperature=temperature,
                num_return_sequences=n_traces, top_p=0.95,
                pad_token_id=self.tok.pad_token_id, use_cache=True)
        if self.probe and len(self._chunks) >= 2:
            embs = torch.stack(self._chunks[1:], dim=0).mean(dim=0).numpy()
            scores = self.probe.predict_proba(embs)[:, 1].tolist()
        else:
            scores = [1.0] * n_traces
        answers = [self.extract_answer(self.tok.decode(out[i, ids.shape[1]:], skip_special_tokens=True)) for i in range(n_traces)]
        weighted = defaultdict(float)
        for a, s in zip(answers, scores):
            if a: weighted[a] += s
        return max(weighted, key=weighted.get) if weighted else None'''
(OUT / 'hyperqwen.py').write_text(WRAPPER_CODE)

summary = {
    'model': MODEL_ID,
    'training': {
        'dataset': 's1K-1.1',
        'n_samples': len(train_ds),
        'epochs': EPOCHS,
        'lora_r': LORA_R, 'lora_alpha': LORA_ALPHA,
        'target_modules': TARGET_MODULES,
        'final_loss': float(result.training_loss),
    },
    'eval': {
        'n_prompts': len(eval_set),
        'seed': EVAL_SEED,
        'results': {k: float(np.mean([r[1] for r in v])) for k, v in results.items()},
        'deltas_pp': {
            'pwmv_on_base': float((b_p-b_g)*100),
            'pwmv_on_adapter': float((a_p-a_g)*100),
            'lora_on_greedy': float((a_g-b_g)*100),
            'lora_on_pwmv': float((a_p-b_p)*100),
            'full_stack': float((a_p-b_g)*100),
        },
    },
}
with open(OUT / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

readme = f'''---
license: apache-2.0
base_model: Qwen/Qwen3.6-35B-A3B
tags: [lora, limo, probe-weighted-voting, reasoning]
---

# HyperQwen-A3B: LIMO LoRA + Probe-Weighted Voting

SFT LoRA trained on s1K-1.1 ({len(train_ds)} samples) stacked with PWMV inference wrapper.

## Eval results (n={N_EVAL} SuperGPQA, seed {EVAL_SEED})

| condition | accuracy |
|---|---|
| base greedy | {b_g:.3f} |
| base + PWMV | {b_p:.3f} |
| **adapter greedy** | **{a_g:.3f}** |
| **adapter + PWMV (full stack)** | **{a_p:.3f}** |

## Deltas

- PWMV lift on base: {(b_p-b_g)*100:+.1f}pp
- PWMV lift on adapter: {(a_p-a_g)*100:+.1f}pp
- LoRA lift on greedy: {(a_g-b_g)*100:+.1f}pp
- LoRA lift on PWMV: {(a_p-b_p)*100:+.1f}pp
- **Full stack (adapter+PWMV vs base greedy): {(a_p-b_g)*100:+.1f}pp**

## Components

- `lora_adapter/` — PEFT adapter (r={LORA_R}, alpha={LORA_ALPHA})
- `probe_l11.pkl` — L11 residual LogReg probe (AUROC 0.78)
- `hyperqwen.py` — inference wrapper class

## Training

- Dataset: s1K-1.1
- Epochs: {EPOCHS}, LR: {LR}, cosine warmup 5%
- LoRA targets: {TARGET_MODULES}
- Final training loss: {result.training_loss:.4f}

## Usage

```python
from hyperqwen import HyperQwenWrapper
w = HyperQwenWrapper(
    base_id='Qwen/Qwen3.6-35B-A3B',
    adapter_path='lora_adapter',
    probe_path='probe_l11.pkl')
answer = w.predict(question, options, n_traces=4)
```
'''
with open(OUT / 'README.md', 'w') as f: f.write(readme)

for fn in ['probe_l11.pkl', 'hyperqwen.py', 'summary.json', 'README.md', 'eval_partial.json']:
    fp = OUT / fn
    if fp.exists():
        api.upload_file(path_or_fileobj=str(fp), path_in_repo=fn,
                        repo_id=HF_ADAPTER_REPO, repo_type='model',
                        commit_message=f'upload {fn}')
        print(f'OK {fn}')

print(f'\nDone: https://huggingface.co/{HF_ADAPTER_REPO}')